In [1]:
import sys
import re

sys.path.insert(0, '..')

import pandas as pd
from utils import classify_sentiment, normalize_text

df = pd.read_csv('../data/ai_apps_reviews_100000.csv')

if 'Sentiment' not in df.columns:
    df['Sentiment'] = df['score'].apply(classify_sentiment)

topic_df = df.dropna(subset=['appName', 'content', 'score', 'Sentiment']).copy()
topic_df['content'] = topic_df['content'].apply(normalize_text)
topic_df = topic_df[topic_df['content'] != ''].reset_index(drop=True)

print('Rows:', len(topic_df))
print('Sentiment counts:', topic_df['Sentiment'].value_counts().to_dict())

Rows: 233512
Sentiment counts: {'Positive': 194641, 'Negative': 28565, 'Neutral': 10306}


In [2]:
# Topic modeling (LDA) with Gensim

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim import corpora
from gensim.models import LdaModel



def simple_tokenize(text: str):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    toks = [t for t in text.split() if len(t) >= 3 and t not in ENGLISH_STOP_WORDS]
    return toks

for sent in ['Negative', 'Neutral', 'Positive']:
    subset = topic_df[topic_df['Sentiment'] == sent]
    subset = subset.sample(n=min(1500, len(subset)), random_state=42)

    texts = [simple_tokenize(t) for t in subset['content'].tolist()]
    texts = [t for t in texts if len(t) >= 3]

    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dictionary.doc2bow(t) for t in texts]

    if len(dictionary) < 50 or len(corpus) < 50:
        print(f'Not enough data for LDA on {sent} after filtering.')
        continue

    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=6,
        random_state=42,
        passes=8,
        alpha='auto',
        per_word_topics=False,
    )

    print(f"\nTop topics for {sent} reviews")
    for i in range(6):
        print(f"Topic {i}:", lda.print_topic(i, topn=10))


Top topics for Negative reviews
Topic 0: 0.036*"chatgpt" + 0.021*"good" + 0.016*"real" + 0.016*"send" + 0.015*"app" + 0.015*"didn" + 0.014*"doesn" + 0.014*"ask" + 0.014*"money" + 0.014*"image"
Topic 1: 0.032*"time" + 0.026*"app" + 0.024*"answer" + 0.022*"information" + 0.017*"just" + 0.015*"useless" + 0.014*"wrong" + 0.014*"use" + 0.012*"work" + 0.012*"google"
Topic 2: 0.029*"chat" + 0.025*"just" + 0.024*"time" + 0.022*"app" + 0.017*"limit" + 0.016*"claude" + 0.015*"fix" + 0.015*"use" + 0.014*"answers" + 0.014*"work"
Topic 3: 0.051*"hai" + 0.033*"like" + 0.030*"chat" + 0.029*"chatgpt" + 0.029*"app" + 0.024*"gpt" + 0.020*"bhi" + 0.019*"worst" + 0.018*"good" + 0.015*"make"
Topic 4: 0.067*"number" + 0.065*"phone" + 0.026*"invalid" + 0.018*"keeps" + 0.017*"problem" + 0.017*"working" + 0.017*"poor" + 0.015*"better" + 0.015*"log" + 0.015*"trying"
Topic 5: 0.088*"app" + 0.030*"gemini" + 0.027*"don" + 0.023*"use" + 0.017*"good" + 0.017*"data" + 0.016*"try" + 0.014*"like" + 0.014*"just" + 0.01

In [3]:
# Named Entity Recognition (spaCy)

import spacy
from collections import Counter

nlp = spacy.load('en_core_web_sm')

sample_n = min(2000, len(topic_df))
sample = topic_df.sample(n=sample_n, random_state=42).copy()

entity_counts_by_sent = {s: Counter() for s in ['Negative', 'Neutral', 'Positive']}

for sent, text in zip(sample['Sentiment'].tolist(), sample['content'].tolist()):
    doc = nlp(text)
    for ent in doc.ents:
        key = (ent.text.strip().lower(), ent.label_)
        if key[0]:
            entity_counts_by_sent[sent][key] += 1

print('Top entities by sentiment (sample)')
for sent in ['Negative', 'Neutral', 'Positive']:
    print(f"\n{sent}")
    for (txt, label), cnt in entity_counts_by_sent[sent].most_common(15):
        print(f"  {cnt:>4}  {label:<8}  {txt}")

Top entities by sentiment (sample)

Negative
     8  GPE       ai
     6  ORG       😔
     5  GPE       hai
     4  ORDINAL   first
     4  PERSON    gemini
     4  CARDINAL  one
     4  ORG       gpt
     4  ORG       google
     3  DATE      today
     3  PERSON    claude
     3  GPE       gemini
     3  CARDINAL  two
     2  ORG       gemini
     2  ORG       cerelac
     2  CARDINAL  1

Neutral
     5  ORG       gemini
     5  PERSON    gemini
     4  GPE       gemini
     3  CARDINAL  3
     3  ORG       gpt
     2  PERSON    claude
     2  CARDINAL  4
     2  CARDINAL  5
     2  TIME      4pm
     1  ORG       chatgpt
     1  GPE       ai
     1  PERSON    raha
     1  CARDINAL  1
     1  TIME      few seconds
     1  CARDINAL  2

Positive
    25  GPE       ai
    17  ORG       gpt
    15  GPE       hai
    11  PERSON    claude
    10  CARDINAL  one
    10  ORG       ai
     6  ORG       chatgpt
     6  ORG       gemini
     5  GPE       gemini
     5  CARDINAL  ❤
     4  PERSON 